In [16]:
require 'gd'

# this function show the image on the notebook
def render(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

:render

In [17]:
require 'gd'

# ═════════════════════════════════════════════════════════════════
#  SafeMath — expression evaluator using Ruby's Math module
# ═════════════════════════════════════════════════════════════════
module SafeMath
  class EvalContext
    include Math

    def pi; Math::PI; end
    def e;  Math::E;  end

    def initialize(vars)
      vars.each do |k, v|
        val = v.to_f
        define_singleton_method(k) { val }
      end
    end

    def ln(v);    Math.log(v);   end
    def log(v);   Math.log(v);   end
    def log10(v); Math.log10(v); end
    def log2(v);  Math.log2(v);  end
    def abs(v);   v.abs;         end
    def cbrt(v);  Math.cbrt(v);  end

    def eval_expr(expr)
      result = eval(expr)
      return nil unless result.is_a?(Numeric)
      return nil if result.respond_to?(:infinite?) && result.infinite?
      return nil if result.respond_to?(:nan?)      && result.nan?
      result.to_f
    end
  end

  def self.evaluate(expr, vars = {})
    EvalContext.new(vars).eval_expr(expr)
  rescue Math::DomainError, ZeroDivisionError, FloatDomainError
    nil
  rescue StandardError
    nil
  end
end


# ═════════════════════════════════════════════════════════════════
#  Plot3D — perspective 3D plotter built on ruby-libgd
#
#  Supports:
#    surface(expr)   — z = f(x,y) with viridis-style colormap shading
#    bars(data)      — 3D bar chart with flat face shading
#
#  Usage — surface:
#    plot = Plot3D.new
#    plot.surface("sin(sqrt(x**2 + y**2))")
#    plot.render("/work/surface.png")
#
#  Usage — 3D bars:
#    data = { "Ruby" => 4.2, "Python" => 9.1, "Julia" => 6.7 }
#    plot = Plot3D.new
#    plot.bars(data)
#    plot.render("/work/bars.png")
#
#  Constructor options (all optional):
#    width:      canvas width  in pixels  (default: 700)
#    height:     canvas height in pixels  (default: 520)
#    xmin/xmax:  x world range            (default: -5..5)
#    ymin/ymax:  y world range            (default: -5..5)
#    zmin/zmax:  z world range, auto if nil
#    resolution: grid steps for surface   (default: 40)
#    yaw:        horizontal rotation deg  (default: 35)
#    pitch:      vertical   rotation deg  (default: 30)
#    fov:        perspective field angle  (default: 500)
#    bg:         [r,g,b] background       (default: [250,250,248])
# ═════════════════════════════════════════════════════════════════
class Plot3D

  # ── Viridis-inspired colormap (8 stops, dark-blue → yellow) ──
  VIRIDIS = [
    [ 68,   1,  84],
    [ 72,  35, 116],
    [ 64,  67, 135],
    [ 52,  94, 141],
    [ 41, 120, 142],
    [ 32, 144, 140],
    [ 34, 167, 132],
    [ 68, 190, 112],
    [121, 209,  81],
    [189, 222,  38],
    [253, 231,  37],
  ].freeze

  def initialize(
    width:      700,
    height:     520,
    xmin:      -5.0,
    xmax:       5.0,
    ymin:      -5.0,
    ymax:       5.0,
    zmin:       nil,
    zmax:       nil,
    resolution: 40,
    yaw:        35.0,
    pitch:      30.0,
    fov:        500.0,
    bg:         [250, 250, 248]
  )
    @width      = width
    @height     = height
    @xmin       = xmin.to_f;  @xmax = xmax.to_f
    @ymin       = ymin.to_f;  @ymax = ymax.to_f
    @zmin       = zmin&.to_f; @zmax = zmax&.to_f
    @resolution = resolution
    @fov        = fov.to_f
    @bg         = bg

    # Convert degrees to radians
    yr = yaw   * Math::PI / 180.0
    pr = pitch * Math::PI / 180.0

    # Rotation matrices — Ry (yaw) then Rx (pitch)
    @cos_y = Math.cos(yr); @sin_y = Math.sin(yr)
    @cos_p = Math.cos(pr); @sin_p = Math.sin(pr)

    @mode  = nil
    @expr  = nil
    @data  = nil
  end

  # ── Public API ───────────────────────────────────────────────

  def surface(expr)
    @mode = :surface
    @expr = expr
    self
  end

  def bars(data)
    @mode = :bars
    @data = data
    self
  end

  def render(path = "graph.png")
    img = case @mode
          when :surface then render_surface
          when :bars    then render_bars
          else raise "Call .surface(expr) or .bars(data) before .render"
          end
    img.save("/work/#{path}")
  end

  private

  # ════════════════════════════════════════════════════════════
  #  SURFACE RENDERING
  # ════════════════════════════════════════════════════════════
  def render_surface
    img = new_image

    # 1. Sample the grid
    grid = sample_grid

    # 2. Auto z-range if not provided
    zs = grid.flatten.compact
    zmin = @zmin || zs.min
    zmax = @zmax || zs.max
    zrange = (zmax - zmin).nonzero? || 1.0

    # 3. Build patches: each patch = 4 corner 3D points + average z
    patches = []
    nx = @resolution
    ny = @resolution

    nx.times do |i|
      ny.times do |j|
        z00 = grid[i][j];     next unless z00
        z10 = grid[i+1]&.[](j);   next unless z10
        z01 = grid[i][j+1];   next unless z01
        z11 = grid[i+1]&.[](j+1); next unless z11

        x0 = @xmin + i.to_f       / nx * (@xmax - @xmin)
        x1 = @xmin + (i + 1).to_f / nx * (@xmax - @xmin)
        y0 = @ymin + j.to_f       / ny * (@ymax - @ymin)
        y1 = @ymin + (j + 1).to_f / ny * (@ymax - @ymin)

        pts3d = [[x0,y0,z00],[x1,y0,z10],[x1,y1,z11],[x0,y1,z01]]
        avg_z = (z00 + z10 + z01 + z11) / 4.0

        # Painter's algorithm key: average projected depth
        proj   = pts3d.map { |p| rotate3d(*p) }
        avg_depth = proj.sum { |p| p[2] } / 4.0

        patches << { pts3d: pts3d, proj: proj, avg_z: avg_z, depth: avg_depth }
      end
    end

    # 4. Sort back-to-front (painter's algorithm)
    patches.sort_by! { |p| -p[:depth] }

    # 5. Draw each patch
    patches.each do |patch|
      t    = (patch[:avg_z] - zmin) / zrange
      color = colormap(t)
      c     = img_color(img, *color)
      pts2d = patch[:proj].map { |p| project2d(*p) }
      draw_filled_polygon(img, pts2d, c)

      # Thin edge for depth clarity
      edge_c = img_color(img, *darken(color, 0.6))
      draw_polygon_outline(img, pts2d, edge_c)
    end

    img
  end

  # ════════════════════════════════════════════════════════════
  #  BAR CHART RENDERING
  # ════════════════════════════════════════════════════════════
  def render_bars
    img    = new_image
    labels = @data.keys
    values = @data.values.map(&:to_f)
    n      = labels.size

    vmax  = (@zmax || values.max).to_f
    vmax  = 1.0 if vmax.zero?

    # Bar geometry in world space
    bar_w  = 0.6   # width  in x
    bar_d  = 0.6   # depth  in y
    gap    = 1.2   # center spacing

    # World x range fits all bars centered
    total_w = (n - 1) * gap
    x_start = -total_w / 2.0

    # Base color palette for bars (one per bar, 3 face shades each)
    palette = [
      [78, 121, 167], [242, 142,  43], [ 89, 161,  79],
      [237,  88,  84], [176, 122, 161], [255, 157, 167],
      [118, 183, 178], [156, 117,  95], [186, 176, 171],
    ]

    n.times do |i|
      val    = values[i]
      height = (val / vmax) * 8.0   # scale to world z units
      cx     = x_start + i * gap
      cy     = 0.0

      x0 = cx - bar_w / 2.0;  x1 = cx + bar_w / 2.0
      y0 = cy - bar_d / 2.0;  y1 = cy + bar_d / 2.0
      z0 = 0.0;                z1 = height

      base_color = palette[i % palette.size]

      # 6 faces of the box — each with its own shade
      # Only draw visible faces given our camera angle (front, top, right)
      faces = [
        # [corners_3d, shade_factor]
        # Front face  (y = y0)
        { pts: [[x0,y0,z0],[x1,y0,z0],[x1,y0,z1],[x0,y0,z1]], shade: 0.85 },
        # Right face  (x = x1)
        { pts: [[x1,y0,z0],[x1,y1,z0],[x1,y1,z1],[x1,y0,z1]], shade: 0.65 },
        # Top face    (z = z1)
        { pts: [[x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1]], shade: 1.00 },
      ]

      # Sort faces back-to-front
      faces_proj = faces.map do |face|
        proj     = face[:pts].map { |p| rotate3d(*p) }
        avg_dep  = proj.sum { |p| p[2] } / proj.size.to_f
        pts2d    = proj.map { |p| project2d(*p) }
        { pts2d: pts2d, shade: face[:shade], depth: avg_dep }
      end
      faces_proj.sort_by! { |f| -f[:depth] }

      faces_proj.each do |face|
        shaded = shade_color(base_color, face[:shade])
        c      = img_color(img, *shaded)
        draw_filled_polygon(img, face[:pts2d], c)
        edge_c = img_color(img, *darken(shaded, 0.7))
        draw_polygon_outline(img, face[:pts2d], edge_c)
      end

      # Label below the bar base
      base_pts = [[x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0]]
        .map { |p| project2d(*rotate3d(*p)) }
      lx = base_pts.sum { |p| p[0] } / 4
      ly = base_pts.sum { |p| p[1] } / 4 + 14
      # (text rendering requires a font path — skipped here, add via img.text if available)
    end

    img
  end

  # ════════════════════════════════════════════════════════════
  #  3D MATH
  # ════════════════════════════════════════════════════════════

  # Rotate point by yaw (around Z) then pitch (around X)
  def rotate3d(x, y, z)
    # Yaw — rotate around Z axis
    x1 =  x * @cos_y + y * @sin_y
    y1 = -x * @sin_y + y * @cos_y
    z1 =  z

    # Pitch — rotate around X axis
    x2 =  x1
    y2 =  y1 * @cos_p - z1 * @sin_p
    z2 =  y1 * @sin_p + z1 * @cos_p

    [x2, y2, z2]
  end

  # Perspective projection — returns [screen_x, screen_y]
  def project2d(x, y, z)
    # Translate camera back so scene is in front
    cam_z = z - 20.0
    denom = (cam_z.abs < 0.001) ? 0.001 : cam_z.abs

    sx = (@fov * x / denom + @width  / 2.0).to_i
    sy = (@fov * y / denom + @height / 2.0).to_i
    [sx, sy]
  end

  # Sample z = f(x,y) grid  — returns (resolution+1)×(resolution+1) array
  def sample_grid
    n    = @resolution
    grid = Array.new(n + 1) { Array.new(n + 1) }
    (n + 1).times do |i|
      (n + 1).times do |j|
        x = @xmin + i.to_f / n * (@xmax - @xmin)
        y = @ymin + j.to_f / n * (@ymax - @ymin)
        grid[i][j] = SafeMath.evaluate(@expr, x: x, y: y)
      end
    end
    grid
  end

  # ════════════════════════════════════════════════════════════
  #  COLOR HELPERS
  # ════════════════════════════════════════════════════════════

  # Map t ∈ [0,1] through the viridis colormap
  def colormap(t)
    t     = t.clamp(0.0, 1.0)
    stops = VIRIDIS
    idx   = t * (stops.size - 1)
    lo    = idx.floor.clamp(0, stops.size - 2)
    hi    = lo + 1
    frac  = idx - lo

    lo_c = stops[lo]; hi_c = stops[hi]
    r = (lo_c[0] + frac * (hi_c[0] - lo_c[0])).round.clamp(0, 255)
    g = (lo_c[1] + frac * (hi_c[1] - lo_c[1])).round.clamp(0, 255)
    b = (lo_c[2] + frac * (hi_c[2] - lo_c[2])).round.clamp(0, 255)
    [r, g, b]
  end

  # Multiply each channel by factor (for face shading)
  def shade_color(rgb, factor)
    rgb.map { |c| (c * factor).round.clamp(0, 255) }
  end

  def darken(rgb, factor)
    shade_color(rgb, factor)
  end

  def img_color(img, r, g, b)
    GD::Color.rgb(r, g, b)
  end

  # ════════════════════════════════════════════════════════════
  #  DRAWING HELPERS
  # ════════════════════════════════════════════════════════════

  def new_image
    img = GD::Image.new(@width, @height)
    img.filled_rectangle(0, 0, @width - 1, @height - 1, GD::Color.rgb(*@bg))
    img
  end

  def draw_filled_polygon(img, pts2d, color)
    return if pts2d.size < 3
    img.filled_polygon(pts2d, color)
  rescue StandardError
    nil
  end

  def draw_polygon_outline(img, pts2d, color)
    return if pts2d.size < 2
    pts2d.each_cons(2) { |a, b| safe_line(img, a, b, color) }
    safe_line(img, pts2d.last, pts2d.first, color)
  end

  def safe_line(img, a, b, color)
    x1 = a[0].clamp(0, @width  - 1)
    y1 = a[1].clamp(0, @height - 1)
    x2 = b[0].clamp(0, @width  - 1)
    y2 = b[1].clamp(0, @height - 1)
    img.line(x1, y1, x2, y2, color)
  rescue StandardError
    nil
  end
end

(irb):78: warning: already initialized constant #<Class:0x00007f4f3ac3b7d0>::Plot3D::VIRIDIS
(irb):78: warning: previous definition of VIRIDIS was here


:safe_line

In [18]:
# ═════════════════════════════════════════════════════════════════
#  CELL 1 — Surface plot: z = sin(sqrt(x² + y²))
# ═════════════════════════════════════════════════════════════════
surface = Plot3D.new(
  width:      700,
  height:     520,
  xmin:      -5.0,  xmax: 5.0,
  ymin:      -5.0,  ymax: 5.0,
  resolution: 40,
  yaw:        35.0,
  pitch:      30.0,
  fov:        500.0
)
surface.surface("sin(sqrt(x**2 + y**2))")
name = "surface.png"
surface.render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 2 — 3D bar chart
# ═════════════════════════════════════════════════════════════════
data = {
  "Ruby"    => 4.2,
  "Python"  => 9.1,
  "Julia"   => 6.7,
  "R"       => 5.8,
  "Rust"    => 3.9,
}

bars = Plot3D.new(
  width:  700,
  height: 520,
  yaw:    25.0,
  pitch:  25.0,
  fov:    480.0
)
bars.bars(data)
file_name = "bars.png"
bars.render(file_name)
render(file_name)

"<img src='surface.png' />"

"<img src='bars.png' />"

SignalException: SIGTERM